In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import io, re, warnings
warnings.filterwarnings('ignore')

with open('Test.csv','r',encoding='utf-8',errors='replace') as f:
    lines = f.readlines()
header = lines[0].strip().rstrip(';').rstrip(',')
cleaned = [header]
for line in lines[1:]:
    line = line.rstrip('\n').rstrip('\r')
    if not line.strip(): continue
    if line.lstrip().startswith('"'):
        line = re.sub(r';+\s*$','',line.strip())
        if line.endswith('"'):
            line = line[1:-1]
            line = line.replace('""','"')
    cleaned.append(line)
df = pd.read_csv(io.StringIO('\n'.join(cleaned)),low_memory=False,on_bad_lines='skip')
df.columns = [c.rstrip(';') for c in df.columns]
for k in ['Qty','Total']:
    df[k] = pd.to_numeric(df[k].astype(str).str.replace(',',''),errors='coerce')
df['Tanggal'] = pd.to_datetime(df['Tanggal'],format='mixed',dayfirst=True,errors='coerce')
df = df.dropna(subset=['Tanggal','Total'])
df = df[df['Qty'] > 0].copy()
print(f'Data siap: {len(df):,} baris')

In [ ]:
def abc_classify(data, kolom_nama, kolom_nilai):
    result = data.groupby(kolom_nama)[kolom_nilai].sum().sort_values(ascending=False).reset_index()
    result.columns = [kolom_nama, 'Omzet']
    total = result['Omzet'].sum()
    result['Omzet_Kumulatif'] = result['Omzet'].cumsum()
    result['Pct_Kumulatif']   = result['Omzet_Kumulatif'] / total * 100
    result['Kelas'] = result['Pct_Kumulatif'].apply(
        lambda x: 'A' if x <= 80 else ('B' if x <= 95 else 'C')
    )
    return result

# ABC Customer
abc_cust = abc_classify(df, 'N. Customer', 'Total')
print('=== ABC Customer ===')
print(abc_cust.groupby('Kelas').agg(Jumlah=('N. Customer','count'), Total_Omzet=('Omzet','sum')).assign(
    Pct_Customer=lambda x: x['Jumlah']/x['Jumlah'].sum()*100,
    Pct_Omzet=lambda x: x['Total_Omzet']/x['Total_Omzet'].sum()*100
).round(1).to_string())

# ABC Produk
abc_prod = abc_classify(df, 'N. Barang', 'Total')
print('\n=== ABC Produk ===')
print(abc_prod.groupby('Kelas').agg(Jumlah=('N. Barang','count'), Total_Omzet=('Omzet','sum')).assign(
    Pct_Produk=lambda x: x['Jumlah']/x['Jumlah'].sum()*100,
    Pct_Omzet=lambda x: x['Total_Omzet']/x['Total_Omzet'].sum()*100
).round(1).to_string())

In [ ]:
# Pareto Chart Customer
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, data, title in zip(axes, [abc_cust, abc_prod], ['Customer','Produk']):
    x = range(len(data))
    color_map = {'A':'#2ecc71','B':'#f39c12','C':'#e74c3c'}
    colors = data['Kelas'].map(color_map)
    ax.bar(x, data['Omzet']/1e6, color=colors, alpha=0.7)
    ax2 = ax.twinx()
    ax2.plot(x, data['Pct_Kumulatif'], color='navy', linewidth=2)
    ax2.axhline(80, color='green', linestyle='--', linewidth=1, label='80%')
    ax2.axhline(95, color='orange', linestyle='--', linewidth=1, label='95%')
    ax2.set_ylabel('Kumulatif %')
    ax2.set_ylim(0, 110)
    ax2.legend(loc='center right')
    ax.set_title(f'Pareto — ABC {title}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Omzet (Juta Rp)')
    ax.set_xticks([])
    # Legend kelas
    from matplotlib.patches import Patch
    legend = [Patch(color=c, label=f'Kelas {k}') for k,c in color_map.items()]
    ax.legend(handles=legend, loc='upper right')

plt.tight_layout()
plt.savefig('abc_pareto.png', dpi=120)
plt.show()

In [ ]:
abc_cust.to_csv('abc_customer.csv', index=False)
abc_prod.to_csv('abc_produk.csv', index=False)
print('Tersimpan: abc_customer.csv, abc_produk.csv')
print(f"\nKelas A Customer: {len(abc_cust[abc_cust['Kelas']=='A'])} pelanggan")
print(f"Kelas A Produk  : {len(abc_prod[abc_prod['Kelas']=='A'])} produk")